# MuscleMimic full-body policy tutorial (simple notebook flow)

This is a minimal educational example for running a policy with a trajectory reference.

## Flow

1. Set a checkpoint + motion path.
2. Load model, trajectory, and policy.
3. Run rollout offscreen and preview MP4 inline.
4. (Optional) try interactive MuJoCo viewer.

No CLI wrappers are used.


In [ ]:
import os
os.environ["JAX_PLATFORM_NAME"] = 'cpu'

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import mujoco
import numpy as np
from IPython.display import Markdown, Video, display

from myosuite.integrations.musclemimic.fullbody_checkpoint_io import resolve_checkpoint_ref
from myosuite.integrations.musclemimic.fullbody_local_policy import (
    FullbodyObsAdapter,
    LocalPolicyRunner,
    has_local_policy_artifacts,
    load_local_policy_artifacts,
    read_checkpoint_config_metadata,
)
from myosuite.integrations.musclemimic.fullbody_model import (
    compile_musclemimic_fullbody_mjmodel,
    default_musclemimic_fullbody_config,
)
from myosuite.core.trajectory_io import load_motion_clip, resolve_motion_path

# ---- Edit these values ----
CHECKPOINT_REF = "hf://amathislab/mm-10m-2"
MOTION_PATH = "KIT/314/walking_medium09_poses"
N_STEPS = 300
EVAL_SEED = 0
FRAME_SKIP = 5
VIDEO_PATH = Path("musclemimic_fullbody_policy_preview.mp4")
REQUESTED_WIDTH = 960
REQUESTED_HEIGHT = 540

print("Checkpoint:", CHECKPOINT_REF)
print("Motion:", MOTION_PATH)
print("Steps:", N_STEPS)


In [ ]:
# 1) Resolve checkpoint
checkpoint = resolve_checkpoint_ref(CHECKPOINT_REF)
checkpoint_root = checkpoint.local_path
if not has_local_policy_artifacts(checkpoint_root):
    raise RuntimeError("Checkpoint is missing local policy artifacts (train_state/config).")

# 2) Build full-body MuJoCo model
cfg = default_musclemimic_fullbody_config()
model, _spec, xml_path = compile_musclemimic_fullbody_mjmodel(cfg)

# 3) Load trajectory clip
motion_file = resolve_motion_path(MOTION_PATH, env_name="MyoFullBody")
clip = load_motion_clip(motion_file, expected_nq=model.nq, expected_nv=model.nv)

# 4) Build obs adapter from checkpoint goal_params + load policy
cfg_meta = read_checkpoint_config_metadata(checkpoint_root)
goal_params = (
    cfg_meta.get("experiment", {})
    .get("env_params", {})
    .get("goal_params", {})
)
obs_adapter = FullbodyObsAdapter(model=model, clip=clip, goal_params=goal_params)

artifacts = load_local_policy_artifacts(checkpoint_root)
policy = LocalPolicyRunner(
    artifacts=artifacts,
    stochastic=False,
    seed=EVAL_SEED,
    frame_skip=FRAME_SKIP,
    obs_adapter=obs_adapter,
)

print("Resolved checkpoint:", checkpoint_root)
print("Model dims (nq, nv, nu):", (model.nq, model.nv, model.nu))
print("Motion file:", motion_file)
print("Frames in clip:", int(clip.qpos.shape[0]))


In [ ]:
def make_initialized_data(model: mujoco.MjModel, clip) -> mujoco.MjData:
    data = mujoco.MjData(model)
    data.qpos[:] = clip.qpos[0]
    if clip.qvel is not None and clip.qvel.shape[0] > 0:
        data.qvel[:] = clip.qvel[0]
    mujoco.mj_forward(model, data)
    return data


def safe_render_size(model: mujoco.MjModel, width: int, height: int) -> tuple[int, int]:
    max_w = int(getattr(model.vis.global_, "offwidth", 0))
    max_h = int(getattr(model.vis.global_, "offheight", 0))
    w = min(width, max_w) if max_w > 0 else width
    h = min(height, max_h) if max_h > 0 else height
    return max(1, int(w)), max(1, int(h))

def safe_write_video(video_path, frames, fps):
    try:
        import imageio
    except ImportError as err:
        raise ImportError("Install imageio: pip install imageio") from err
    try:
        with imageio.get_writer(video_path, fps=fps) as video:
            for frame in frames:
                video.append_data(frame)
        return True
    except Exception as e:
        print(f"Skipping video export for {video_path}: {e}")
        return False

def run_offscreen_rollout() -> dict[str, float | int | str]:

    data = make_initialized_data(model, clip)
    steps = min(int(N_STEPS), int(clip.qpos.shape[0]))
    width, height = safe_render_size(model, REQUESTED_WIDTH, REQUESTED_HEIGHT)

    renderer = mujoco.Renderer(model, height=height, width=width)
    frames: list[np.ndarray] = []
    qpos_l2_errors: list[float] = []
    try:
        for i in range(steps):
            action = policy.action_for(data, clip, i)
            policy.step(model, data, action)

            target_qpos = clip.qpos[min(i, clip.qpos.shape[0] - 1)]
            qpos_l2_errors.append(float(np.linalg.norm(np.asarray(data.qpos) - np.asarray(target_qpos))))

            renderer.update_scene(data)
            frames.append(np.asarray(renderer.render(), dtype=np.uint8))
    finally:
        renderer.close()

    video_path = VIDEO_PATH.expanduser().resolve()
    video_path.parent.mkdir(parents=True, exist_ok=True)
    fps = max(1, int(round(1.0 / float(model.opt.timestep))))
    safe_write_video(str(video_path), np.asarray(frames, dtype=np.uint8), fps)

    return {
        "steps": steps,
        "mean_qpos_l2": float(np.mean(qpos_l2_errors)),
        "max_qpos_l2": float(np.max(qpos_l2_errors)),
        "video_path": str(video_path),
        "width": width,
        "height": height,
    }


metrics = run_offscreen_rollout()

display(Markdown(
    f"""
### Offscreen rollout complete
- Steps: `{metrics['steps']}`
- Render size: `{metrics['width']} x {metrics['height']}`
- Mean qpos L2 error: `{metrics['mean_qpos_l2']:.6f}`
- Max qpos L2 error: `{metrics['max_qpos_l2']:.6f}`
- Video: `{metrics['video_path']}`
"""
))

display(Video(metrics["video_path"], embed=True, html_attributes="controls loop"))


In [ ]:
# Optional interactive preview (often unavailable in notebook runtimes on macOS).
# You can skip this cell.

try:
    data = make_initialized_data(model, clip)
    preview_steps = min(int(N_STEPS), 200)
    with mujoco.viewer.launch_passive(model, data) as viewer:
        for i in range(preview_steps):
            action = policy.action_for(data, clip, i)
            policy.step(model, data, action)
            viewer.sync()
    print(f"Interactive preview completed for {preview_steps} steps.")
except RuntimeError as err:
    print("Interactive viewer not available in this runtime.")
    print(f"Reason: {err}")

## Troubleshooting

- If checkpoint loading fails, install `orbax-checkpoint` in this environment.
- If MP4 export fails, install `imageio`.
- If interactive viewer fails on macOS, prefer offscreen rendering in notebooks.

To try another trajectory, only change `MOTION_PATH` and re-run from the load cell.